## Import and Setup

In [2]:
import os
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tdc.multi_pred import DrugRes
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

## Load data

In [ ]:
data = DrugRes(name='GDSC2', path='../data')

Downloading...


In [ ]:
df = data.get_data()
df.head()

# Preprocess

In [ ]:
def smiles_to_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(1024, dtype=np.int8)
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024), dtype=np.int8)
    except Exception:
        return np.zeros(1024, dtype=np.int8)

# Train/Val/Test split

In [ ]:
split = data.get_split(
    method='cold_split',
    column_name='Cell Line_ID',
    seed=42,
    frac=[0.7, 0.1, 0.2],
)

print(f"Split xong: Train ({len(split['train'])}), Val ({len(split['valid'])}), Test ({len(split['test'])})")

In [ ]:
train_unique_cells = split['train'].drop_duplicates(subset=['Cell Line_ID'])
train_matrix = np.array([np.array(val, dtype=np.float32) for val in train_unique_cells['Cell Line'].values])

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_matrix)

pca = PCA(n_components=0.95, random_state=42)
pca.fit(train_scaled)

print(f"PCA hoàn tất: Giữ lại {pca.n_components_} chiều, giải thích {np.sum(pca.explained_variance_ratio_):.2%} phương sai.")

all_unique_df = df.drop_duplicates(subset=['Cell Line_ID'])
all_ids = all_unique_df['Cell Line_ID'].values
all_matrix = np.array([np.array(val, dtype=np.float32) for val in all_unique_df['Cell Line'].values])

all_pca_feats = pca.transform(scaler.transform(all_matrix))
cell_pca_map = {cid: feat for cid, feat in zip(all_ids, all_pca_feats)}

del train_matrix, train_scaled, all_matrix, all_unique_df, all_pca_feats
gc.collect()

In [ ]:
def build_dataset(target_df, pca_map):
    drug_feats = np.array([smiles_to_fp(s) for s in target_df['Drug'].values], dtype=np.int8)
    cell_feats = np.array([pca_map[cid] for cid in target_df['Cell Line_ID'].values], dtype=np.float32)
    X = np.hstack([drug_feats, cell_feats])
    y = target_df['Y'].values
    del drug_feats, cell_feats
    gc.collect()
    return X, y

print("Đang tạo ma trận X, y...")
X_train, y_train = build_dataset(split['train'], cell_pca_map)
X_val, y_val = build_dataset(split['valid'], cell_pca_map)
X_test, y_test = build_dataset(split['test'], cell_pca_map)

del split, df
gc.collect()
print(f"Hoàn tất. Kích thước X_train: {X_train.shape}")

# Train Linear Regression

In [ ]:
final_model = LinearRegression()
final_model.fit(X_train, y_train)

val_pred = final_model.predict(X_val)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_r2 = r2_score(y_val, val_pred)

print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Validation R2: {val_r2:.4f}")

# Evaluate

In [ ]:
y_pred = final_model.predict(X_test)

def get_bootstrap_metrics(y_true, y_pred, n_iterations=100):
    metrics = {
        'r2': [],
        'pearson': [],
        'spearman': [],
        'rmse': [],
    }

    print(f"Đang chạy Bootstrapping ({n_iterations} lần)...")
    for i in range(n_iterations):
        y_true_sample, y_pred_sample = resample(y_true, y_pred, random_state=i)

        metrics['r2'].append(r2_score(y_true_sample, y_pred_sample))
        metrics['pearson'].append(pearsonr(y_true_sample, y_pred_sample)[0])
        metrics['spearman'].append(spearmanr(y_true_sample, y_pred_sample)[0])
        metrics['rmse'].append(np.sqrt(mean_squared_error(y_true_sample, y_pred_sample)))

    return metrics

bootstrapped_results = get_bootstrap_metrics(y_test, y_pred, n_iterations=100)

r2_mean, r2_std = np.mean(bootstrapped_results['r2']), np.std(bootstrapped_results['r2'])
p_mean, p_std = np.mean(bootstrapped_results['pearson']), np.std(bootstrapped_results['pearson'])
s_mean, s_std = np.mean(bootstrapped_results['spearman']), np.std(bootstrapped_results['spearman'])
rmse_mean, rmse_std = np.mean(bootstrapped_results['rmse']), np.std(bootstrapped_results['rmse'])

print("\n" + "=" * 40)
print(f"{'Metric':<12} | {'Mean':<10} | {'STD':<10}")
print("-" * 40)
print(f"{'R2':<12} | {r2_mean:<10.4f} | {r2_std:<10.4f}")
print(f"{'Pearson':<12} | {p_mean:<10.4f} | {p_std:<10.4f}")
print(f"{'Spearman':<12} | {s_mean:<10.4f} | {s_std:<10.4f}")
print(f"{'RMSE':<12} | {rmse_mean:<10.4f} | {rmse_std:<10.4f}")
print("=" * 40)

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.regplot(x=y_test, y=y_pred, scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'color': 'red'})
plt.title(f'Actual vs Predicted\n$R^2$: {r2_mean:.3f} +/- {r2_std:.3f}')
plt.xlabel('Actual log(IC50)')
plt.ylabel('Predicted log(IC50)')

plt.subplot(1, 3, 2)
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, color='purple')
plt.axvline(x=0, color='black', linestyle='--')
plt.title(f'Residuals Distribution\nRMSE: {rmse_mean:.3f} +/- {rmse_std:.3f}')
plt.xlabel('Error')

plt.subplot(1, 3, 3)
sns.histplot(bootstrapped_results['r2'], kde=True, color='green')
plt.axvline(x=r2_mean, color='red', linestyle='--')
plt.title('Bootstrap R2 Distribution')
plt.xlabel('R2')

plt.tight_layout()
plt.show()

In [ ]:
feature_names = [f'drug_fp_{i}' for i in range(1024)] + [f'cell_pca_{i}' for i in range(pca.n_components_)]
coef_importance = pd.Series(np.abs(final_model.coef_), index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
coef_importance.head(20).plot(kind='bar')
plt.title('Top 20 Coefficients by Absolute Value')
plt.ylabel('|Coefficient|')
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../models', exist_ok=True)

model_path = '../models/linear_regression_cold_split.pkl'
joblib.dump(final_model, model_path)
print(f"Đã lưu mô hình tại: {model_path}")